\setcounter{secnumdepth}{0}

# Methylatie bestand #

CCLE_RRBS_TSS1kb_20181022.txt - bevat DNA-methylatiegegevens gemeten rond de Transcription Start Site (TSS, ±1 kb). Dit gebied omvat zowel promoterregio's als het eerste exon, die belangrijk zijn voor genregulatie.


Inleiding:
In dit notebook wordt de DNA-methylatiedata van CCLE voorbereid voor integratie met mutatie- en responsdata. De focus ligt op longkankercellijnen. De preprocessing omvat het verkennen van de datastructuur, het selecteren van relevante cellijnen, kwaliteitscontrole, en het harmoniseren van identifiers zodat de data geschikt is voor verdere analyses en samenvoeging.

### Stap 1. Inlezen en verkennen van de methylatie-data ###

Doel: Het inlezen van de DN-methylatiedata van CCLE en het uivoeren van een eerste verkenning van de dataset.

We bekijken het aantal rijen/kolommen, de eerste en laatste kolomnamen, datatypes en missing values.

In deze stap wordt inzicht verkregen:
- het aantal rijen en kolommen
- de structuur van de dataset
- de aannwezige metadata
- de manier waarop cellijnen zijn gecodeerd
- en de aanwezigheid van eventuele missende waarden.

De dataset bevat methylatiewaarden per CpG-locus voor een groot aantal kankercellijnen. Het is belangrijk om eerst te begrijpen hoe deze cellijnen worden gerepresenteerd voordat filtering op relevante data wordt toegepast.

In [59]:
import pandas as pd
import numpy as np

# aanmaken variabele met de bestandpad
methylatie_file = "../data/raw/CCLE_RRBS_TSS1kb_20181022.txt"

# inlezen
methylatie_df = pd.read_csv(methylatie_file, sep='\t', low_memory=False) # txt, tab seperated

# verkenning
print("Aantal rijen en kolommen:", methylatie_df.shape) # globale info

# eerste en laatste 5 kolommen bekijken
print("Eerste 5 kolommen:", methylatie_df.columns[:5].tolist())
print("Laatste 5 kolommen:", methylatie_df.columns[-5:].tolist())

# datatypes en missende waarden per kolom bekijken
kolom_info_methylatie = pd.DataFrame({ # aanmaken dataframe (met 3 kolommen)
    'kolom': methylatie_df.columns,
    'dtype': [methylatie_df[col].dtype for col in methylatie_df.columns],
    'n_missing': [methylatie_df[col].isna().sum() for col in methylatie_df.columns] # bepaald totaal aantal missende waarden voor elke kolom
})

# eerste 5 en laatste 5 kolommen tonen als voorbeeld
print(kolom_info_methylatie.head(5)) # eerste 5 kolommen
print(kolom_info_methylatie.tail(5)) # laatste 5 kolommen

Aantal rijen en kolommen: (21338, 846)
Eerste 5 kolommen: ['locus_id', 'CpG_sites_hg19', 'avg_coverage', 'DMS53_LUNG', 'SW1116_LARGE_INTESTINE']
Laatste 5 kolommen: ['UO31_KIDNEY', 'SF268_CENTRAL_NERVOUS_SYSTEM', 'SF539_CENTRAL_NERVOUS_SYSTEM', 'SNB75_CENTRAL_NERVOUS_SYSTEM', 'HOP92_LUNG']
                    kolom   dtype  n_missing
0                locus_id  object          0
1          CpG_sites_hg19  object          1
2            avg_coverage  object          0
3              DMS53_LUNG  object          0
4  SW1116_LARGE_INTESTINE  object          0
                            kolom   dtype  n_missing
841                   UO31_KIDNEY  object          0
842  SF268_CENTRAL_NERVOUS_SYSTEM  object          0
843  SF539_CENTRAL_NERVOUS_SYSTEM  object          0
844  SNB75_CENTRAL_NERVOUS_SYSTEM  object          0
845                    HOP92_LUNG  object          0


Waarnemingen:

- De dataset bevat 21338 rijen, waarbij elke rij een CpG-locus representeert, en 846 kolommen.
- De eerste 3 kolommen bevatten metadata:
  - `locus_id`: unieke identifier van de CpG-locus  
  - `CpG_sites_hg19`:genoompositie van de locus (hg19 referentie)
  - `avg_coverage`: gemiddelde sequencingdiepte, een maat voor de betrouwbaarheid van de meting  
- De overige 843 kolommen representeren individuele kankercellijnen.   
- De cellijnen worden niet weergeven met een standaard DepMap-identifier (ACH- of ModelID). In plaats daarvan worden cellijnen geidentificeerd op basis van hun naam, waarbij de kolomtitel zowel de cellijnnaam als het weefseltype bevat (bijvoorbeeled DMS53_LUNG).

Deze verkenninng laat zien dat preprocessing nodig is voordat de methylatiedata kan worden samengevoegd met de andere datasets. In de vervolgstappen zullen de longkankercellen worden geselecteerd, zal een kwaliteitscontrole worden uitgevoerd om niet informatieve loci te verwijderen, en zal er een mapping worden toegepast om de cellijnkolommen te harmoniseren naar standaard DepMap-identifiers (ACH_ID/ModelID).

### Stap 2. Filteren op longkankercellijnen ###

Doel: Alleen de kolommen behouden die longkankercellijnen representeren (_LUNG). Zo wordt er een subset aangemaakt van de relevante data.

Probleem: Methylatiebestand heeft geen DepMap- of ACH-ID's, maar gebruikt kolomnamen die bestaan uit cellijnnaam + weefseltype (zoals DMS53_LUNG, SW1116_LARGE_INTESTINE). Elke kolom is een cellijn en elke rij is een metylatie-feature (CpG-site).

In plaats van filteren op rijen, moeten de kolommen worden geselecteerd die bij longkanker horen (dus de kolommen waarvan de naam "_LUNG" bevat.

In [60]:
# filter kolommen die longkankercellijnen representeren
lung_columns = [col for col in methylatie_df.columns if 'lung' in col.lower()]

# maak dataframe met longkankercellijnen + metadata
lung_methylatie_df = methylatie_df[['locus_id', 'CpG_sites_hg19', 'avg_coverage'] + lung_columns]

print(f"Aantal longkankercellijnen: {len(lung_columns)}")
print("Vorm van longkankermethylatie matrix:", lung_methylatie_df.shape)
print(lung_methylatie_df.head(5))
      

Aantal longkankercellijnen: 153
Vorm van longkankermethylatie matrix: (21338, 156)
                    locus_id  \
0  SGIP1_1_66998638_66999638   
1  SGIP1_1_66998251_66999251   
2  AZIN2_1_33545713_33546713   
3  AZIN2_1_33546778_33547778   
4  AGBL4_1_50489626_50490626   

                                      CpG_sites_hg19 avg_coverage DMS53_LUNG  \
0  1:66998970;1:66998973;1:66998993;1:66999404;1:...        25.00    0.00000   
1                   1:66998970;1:66998973;1:66998993         8.27    0.00000   
2  1:33546151;1:33546209;1:33546210;1:33546385;1:...       326.58    0.00729   
3  1:33546783;1:33546788;1:33546795;1:33546797;1:...       480.54    0.22276   
4  1:50489632;1:50489641;1:50489671;1:50489677;1:...       263.36    0.00000   

  NCIH1184_LUNG NCIH2227_LUNG RERFLCAD2_LUNG NCIH2347_LUNG NCIH2087_LUNG  \
0       0.11864       0.04545            NaN       0.25000       0.86441   
1       0.22579           NaN            NaN           NaN       0.97871   
2       0.22553

Waarnemingen:
- na filtering op kolomnamen die `_LUNG` bevatten zijn 153 longkankercellijnen geselecteerd.
- de resulterende matrix bevat 21338 CpG-loci (rijen) dn 156 kolommen, bestaande uit:
    - 3 metadata kolommen (`locus_id`, `CpG_sites_hg19`, `avg_coverage`)
    - 153 kolommen met methylatiewaarden voor longkankercellijnen
- de aanwezigheid van `NaN`-waarden in sommige cellijnkolommen wijst op ontbrekende methylatiemetingen. Dit zal later worden meegenomen in de kwaliteitscontrole.

De filtering op longkankercellijnen is succesvol toegepast en vormt de basis voor verdere kwaliteitscontrole.

### Stap 3. Quality Control (QC) ###

Doel: Het verbeteren van de betrouwbaarheid van de methylatie data door het verwijderen van CpG-loci met onbetrouwbare metingen of onvoldoende informatiedichtheid.

DNA-methylatiedata bevat vaak ruis als gevolg van lage sequencingdiepte, ontbrekende waarden of loci die in slechts een beperkt aantal cellijnen gemeten zijn. Deze loci verstoren de analyse en leiden tot minder robuuste machine learning-modellen. XXXXX

Daarom wordt in deze stap een reeks kwaliteitscontroles uitgevoerd, waarbij CpG-loci stapsgewijs wordt gefilterd op basis van:
- meetbetrouwbaarheid (coverage)
- mate van ontbrekende waarden
- en variatie over cellijnen
  

#### Coverage Filtering: rijen met lage avg_coverage verwijderen ####


Doel:  
Bij methylatie-metingen varieert de meetdiepte (`avg_coverage`) per CpG-locus. Loci met lage coverage kunnen onbetrouwbare methylatiewaarden bevatten en dragen bij aan ruis in downstream analyses. Door te filteren opcoverage worden alleen loci behouden met voldoende meetbertouwbaarheid.

Aanpak: 
De kolom `avg_coverage` wordt vanuit het tekstbestand ingelezen als string, terwijl het in werkelijkheid een getal is. Om er mee te kunnen rekenen of filteren, moet de kolom worden omgezet naar een numeriek datatype. 

Sommige loci kunnen een ontbrekende of ongeldige coveragewaarde (NaN) hebben. Voor deze loci is het niet mogelijk om de betrouwbaarheid van de meting te beoordelen. Deze rijen leveren geen bruikbare informatie op en worden daarom verwijderd. 

Vervolgens worden alleen loci behouden met een avg_coverage ≥ 10. XXXX bron toevoegen

In [61]:
# maak kopie 
lung_methylatie_qc = lung_methylatie_df.copy()

# converteer avg_coverage naar numeriek
lung_methylatie_qc['avg_coverage'] = pd.to_numeric(lung_methylatie_qc['avg_coverage'], errors ='coerce')

# verwijder rijen met NaN in avg_coverage
lung_methylatie_qc = lung_methylatie_qc.dropna(subset=['avg_coverage'])

# filter loci met avg_coverage >=10
lung_methylation_qc = lung_methylatie_qc[lung_methylatie_qc['avg_coverage']>=10]

# check hoe veel loci overblijven na coverage filtering
print(f"Aantal loci voor filtering op avg coverage: {len(lung_methylatie_df)}")
print(f"Aantal loci na coverage filtering: {len(lung_methylation_qc)}")


Aantal loci voor filtering op avg coverage: 21338
Aantal loci na coverage filtering: 20898


Er zijn slechts een beperkt aantal loci verwijderd. Dit geeft aan dat het merendeel van de CpG-loci met voldoende sequencingdiepte is gemeten. Het resultaat is een subset van CpG-loci die kwalitatief sterk genoeg is voor downstream analyse.

#### Missingness: rijen met teveel ontbrekende waarden verwijderen ####


Doel: Het verwijderen van CpG-loci die in een groot deel van de longkankercellijnen niet gemeten zijn. 

Loci met veel ontbrekende waarden levere geen consistente informatie. 

Aanpak:   
Voor elke locus wordt het percentage ontbrekende waarden berekend over alle longkankercellijnen. Loci met meer dan 10% ontbrekende waarden worden verwijderd. Deze drempel is relatief streng en zorgt ervoor dat de overgebleven loci in vrijwel alle cellijnen beschikbaar zijn, wat de betrouwbaarheid van downstream analyses vergroot.

In [62]:
# maak kopie
lung_methylation_qc = lung_methylation_qc.copy()

# selecteer alleen methylatiekolommen (dus alle cellijnen)
methylation_cols = [col for col in lung_methylation_qc.columns 
                    if col not in ['locus_id', 'CpG_sites_hg19', 'avg_coverage']]

# converteer methylatiekolommen naar numeriek
lung_methylation_qc[methylation_cols] = lung_methylation_qc[methylation_cols].apply(pd.to_numeric, errors='coerce')

# bereken missingness per locus (rij)
lung_methylation_qc['missing_fraction'] = lung_methylation_qc[methylation_cols].isna().mean(axis=1)

# instellen van de missingness drempel
missingness_threshold = 0.10

# filter loci met teveel missende waarden
before = len(lung_methylation_qc)
lung_methylation_qc = lung_methylation_qc[lung_methylation_qc['missing_fraction'] <= missingness_threshold]
after = len(lung_methylation_qc)

# feedback
print(f"Loci vóór missingness filtering: {before}")
print(f"Loci na missingness filtering: {after}")
print(f"Verwijderd door missingness: {before - after}")


Loci vóór missingness filtering: 20898
Loci na missingness filtering: 18449
Verwijderd door missingness: 2449


/var/folders/hf/q9gpk8mn1l34cwvqff64g18c0000gn/T/ipykernel_21628/1362123296.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  lung_methylation_qc['missing_fraction'] = lung_methylation_qc[methylation_cols].isna().mean(axis=1)


Na missingness filtering is ongeveer 12% van de loci is verwijderd. 

#### Variance Filtering ####


Doel:  
Het verwijderen van CpG-loci die weinig variatie laten zien over de longkankercellijnen.

Voorafgaand aan het trainen van het Random Forest-model is variance filtering toegepast om loci met vrijwel constante methylatiewaarden te verwijderen. Per locus is de variatie over alle longkankercellijnen berekend. Loci met een lage variatie dragen weinig bij aan het onderscheiden van cellijnen en kunnen alleen maar ruis introduceren.

In deze analyse is per locus de standaarddeviatie (std) berekend, waarna loci met std <0.05 zijn verwijderd. Deze drempel wordt veelgebruikt in methylatie-studies om niet-informatieve CpG-loci te verwijderen. Technisch wordt dit vertaald naar variance, waarbij std ≥ 0.05 overeenkomt met een variance treshold van 0.0025. 

In [63]:
# maak kopie
lung_methylation_qc = lung_methylation_qc.copy()

# bereken variantie per locus over alle methylatiekolommen
lung_methylation_qc['variance'] = lung_methylation_qc[methylation_cols].var(axis=1, skipna=True)

# stel drempel in (std >= 0.05 is gelijk aan een variance treshold van 0.0025)
variance_threshold = 0.0025

# filter loci met variantie >= drempel
before_var = len(lung_methylation_qc)
lung_methylation_qc = lung_methylation_qc[lung_methylation_qc['variance'] >= variance_threshold]
after_var = len(lung_methylation_qc)

# feedback
print(f"Loci vóór variance filtering: {before_var}")
print(f"Loci na variance filtering: {after_var}")
print(f"Verwijderd door lage variantie: {before_var - after_var}")


Loci vóór variance filtering: 18449
Loci na variance filtering: 10667
Verwijderd door lage variantie: 7782


Na toepassing van variance filtering (std ≥ 0.05 / variance ≥ 0.0025) zijn ongeveer 42% van de loci verwijderd vanwege onvoldoende variatie. Dit is normaal aangezien methylatieprofielen vaak veel CpG-loci bevatten die weinig variatie vertonen (Meng et al., 2010).

### Stap 4. Mapping en transponeren van de methylatie-data ###

Doel: Methylatie-data voorbereiden op integratie met de andere datasets (mutaties en responswaarden).

Daarvoor moeten we de cellijnnamen in de methylatie-dataset eerst worden vertaald naar hun bijbehorende DepMap/ACH-IDs, zodat alle bestanden een gedeelde unieke identifier gebruiken. Vervolgens wordt de dataset getransponeerd zodat cellijnen rijen worden en loci kolommen, wat het juiste formaat is voor downstream analyse en modelbouw.

Aanpak: 

- Metadata inladen:
  De DepMap-annotatie bevat de koppeling tussen CCLE-namen (zoals DMS53_LUNG) en de standaard DepMap/ACH-IDs. Deze dataset wordt ingeladen en beperkt tot de kolommen die nodig zijn voor mapping: `CCLE_ID` --> `DepMapID`.  
- Mapping uitvoeren:
  De methylatiekolommen worden gematcht op de CCLE-celnamen, die via DepMap-annotatie worden gekoppeld aan hun bijbehorende ACH-IDs. Alleen cellijnen waarvoor een geldige DepMap-ID bestaat, worden behouden.
- Transponeren van de dataset:
  Na de mapping wordt het DataFrame omgezet zodat de ACH-IDs de tijen vormen en de methylatie-loci de kolommen. Dit formaat sluit aan bij de andere datasets en maakt latere merge-stappen eenvoudig.


In [64]:
# inlezen van het metadata-bestand en verkenning
metadata_file = "../data/raw/Cell_lines_annotations_20181226.txt"
metadata_df = pd.read_csv(metadata_file, sep='\t', low_memory=False) # txt, tab seperated

print("Metadata ingeladen.")
print(f"Aantal rijen en kolommen: {metadata_df.shape}") 

# belangrijke kolommen selecteren
important_cols = ['CCLE_ID', 'depMapID', 'Name', 'Site_Primary', 
        'Site_Subtype1', 'Site_Subtype2', 'Site_Subtype3', 'Disease']

# alleen tonen als alle kolommen aanwezig zijn
available_cols = [c for c in important_cols if c in metadata_df.columns]

print("\nBeschikbare belangrijke kolommen:")
print(available_cols)

# metadata reduceren tot de kolommen die we nodig hebben voor mapping
metadata_df = metadata_df[['CCLE_ID', 'depMapID']].dropna()

print(f"\nAantal cellijnen bruikbaar voor mapping: {len(metadata_df)}")

# mapping van kolommen naar ACH_ID
methylation_cols = [col for col in lung_methylation_qc.columns 
                    if col not in ['locus_id','CpG_sites_hg19','avg_coverage']]

# mapping: CCLE_ID -> depMapID
ccle2depmap = dict(zip(metadata_df['CCLE_ID'], metadata_df['depMapID']))

# pas mapping toe en behoud alleen kolommen met bestaande mapping
new_cols = []
valid_cols = []
for col in methylation_cols:
    depmap_id = ccle2depmap.get(col)
    if depmap_id:
        new_cols.append(depmap_id)
        valid_cols.append(col)

# maak nieuwe DataFrame met gemapte kolomnamen
meth_mapped_df = lung_methylation_qc[['locus_id','CpG_sites_hg19','avg_coverage'] + valid_cols].copy()
meth_mapped_df.columns = ['locus_id','CpG_sites_hg19','avg_coverage'] + new_cols

print(f"Aantal cellijnen na mapping: {len(new_cols)}")

# transponeren

meth_T = meth_mapped_df.set_index("locus_id").drop(columns=['CpG_sites_hg19','avg_coverage']).T
meth_T.index.name = 'ModelID' # ACH-IDs

print("\nTransponeren voltooid.")
print("Vorm van de getransponeerde methylatie-matrix:", meth_T.shape)


Metadata ingeladen.
Aantal rijen en kolommen: (1461, 33)

Beschikbare belangrijke kolommen:
['CCLE_ID', 'depMapID', 'Name', 'Site_Primary', 'Site_Subtype1', 'Site_Subtype2', 'Site_Subtype3', 'Disease']

Aantal cellijnen bruikbaar voor mapping: 1457
Aantal cellijnen na mapping: 153

Transponeren voltooid.
Vorm van de getransponeerde methylatie-matrix: (153, 10667)


In [57]:
meth_T.head()


locus_id,AZIN2_1_33545713_33546713,AZIN2_1_33546778_33547778,AGBL4_1_50489626_50490626,CLIC4_1_25070759_25071759,SLC45A1_1_8377144_8378144,TGFBR3_1_92351836_92352836,C1orf21_1_184355149_184356149,PRKCZ_1_1980908_1981908,PRKCZ_1_2003900_2004900,PRKCZ_1_2004424_2005424,...,TUBGCP6_22_50683400_50684400,DENND6B_22_50765489_50766489,SCO2_22_50964868_50965868,TYMP_22_50968514_50969514,MIOX_22_50924212_50925212,ADM2_22_50919152_50920152,CPT1B_22_51016506_51017506,MAPK8IP2_22_51038113_51039113,SYCE3_22_51001381_51002381,RPL23AP82_22_51194513_51195513
ModelID,,,,,,,,,,,,,,,,,,,,,
ACH-000698,0.00729,0.22276,0.00000,0.12261,0.06154,0.06495,0.07222,0.01538,0.89951,0.90697,...,0.01696,0.13727,0.00270,0.00000,1.00000,0.13382,0.92511,0.00000,0.00000,0.0000
ACH-000523,0.22553,0.10735,0.07346,0.05042,0.04211,0.12579,0.00134,0.03684,0.82209,0.17693,...,0.02233,0.16354,0.07343,0.01191,0.75902,0.41762,0.99593,0.00374,0.00000,0.0000
ACH-000610,0.04560,0.03326,0.01246,0.12625,0.40403,0.06295,0.00483,0.01948,0.79729,0.20561,...,0.02122,0.11400,0.00370,0.46608,0.74982,0.13184,0.94976,0.01720,0.00722,0.0115
ACH-000774,0.14577,0.09589,0.00531,0.09678,0.18182,0.09434,0.01780,0.00000,0.74443,0.84290,...,0.01180,0.17347,0.11296,0.29867,0.76811,0.04495,0.88168,0.02557,0.03099,0.0000
ACH-000875,0.11487,0.21004,0.07422,0.04676,0.15942,0.11582,0.18736,0.00000,0.92707,0.94621,...,0.05120,0.15221,0.01188,0.01673,0.69831,0.13602,1.00000,0.08504,0.00650,0.0000


### Resultaat van de methylatie-matrix ###

Ter controle zijn de eerste rijen van de getransponeerde methylatiematrix weergegeven. Elke rij representeert één cellijn, geidentificeerd door een `ModelID` (ACH-ID). de kolommen vertegenwoordigen individuele CpG-loci. De CpG-loci volgen de standaard DepMap-namingconventie `GENE_CHROM_START_ENG`, die aangeeft bij welk gen de tile hoort en op welke genomische coördinaten deze is gebaseerd.

De matrix heeft het juiste formaat voor verdere analyses: cellijnen als rijen en methylatiefeatures als kolommen. Deze opgeschoonde, gefilterde en gemapte methylatiematrix wordt opgeslagen en later gebruikt in het final/merge-notebook voor deelvraag 1. 

In [58]:
# sla dataframe op als pickle(houdt types en index exact hetzelfde)
meth_T.to_pickle("meth_T.pkl")